In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if 'notebooks' in str(project_root):
    while project_root.name != 'artificial-causal-inference' and project_root.parent != project_root:
        project_root = project_root.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "notebooks" / "visualization"))

from IPython.display import display, Image as IPImage
from src.dataset.get_dataset import load_dataset
from visualize import generate_comparison

# Treatment Comparison Animation

Side-by-side animated PNG comparing video clips across two treatment conditions.  
Green border = grooming activity on that frame.

**Modes:**
- `"diagnostic"` — original frames, white background, batch/position labels
- `"final"` — circular crop, black background, clean

In [ ]:
# ============ CONFIGURATION ============
subject = "ants"
version = "v3"
treatment_1 = 2        # control
treatment_2 = 9        # treatment to compare
mode = "diagnostic"          # "diagnostic" or "final"

grid_rows = 5
grid_cols = 4
start_frame = 1000
duration = 10           # seconds
fps = 5
thumb_size = (100, 100)
show_activity = False    # green border on frames with grooming

dataset_root = project_root / "dataset"
results_root = project_root / "results"

In [ ]:
# Load dataset
ds = load_dataset(subject=subject, version=version, dataset_root=str(dataset_root))
df = ds.to_pandas()

print(f"Dataset: {subject}/{version} — {len(df):,} frames, {df['observation_id'].nunique()} observations")
print(f"\nTreatment distribution:")
print(df.groupby("T")["observation_id"].nunique().to_string())

frames_dir = dataset_root / subject / version / "frames" / "full"

In [ ]:
# Generate and save animated PNG (APNG — full 24-bit color, unlike GIF's 256-color limit)
preview_root = project_root / "preview"
output_dir = preview_root / subject / version / f"comparison_{mode}"
output_path = output_dir / f"T{treatment_1}_vs_T{treatment_2}.png"

generate_comparison(
    df, frames_dir,
    treatment_1, treatment_2,
    output_path=output_path,
    mode=mode,
    grid_rows=grid_rows, grid_cols=grid_cols,
    start_frame=start_frame,
    duration=duration, fps=fps,
    thumb_size=thumb_size,
    show_activity=show_activity,
)

display(IPImage(filename=str(output_path)))